# Chemical-space exploration

Compare DrugEx and GB-GA molecules generated for the glucocorticoid receptor. All preprocessing and dimensionality reduction is performed jointly while retaining the dataset-of-origin label.


In [ ]:
from pathlib import Path
from glob import glob
import zipfile

import fsspec
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## Load the two generated datasets


In [ ]:
data_root = Path("zenodo_data/data/output_sets/Glucocorticoid_receptor")
dataset_dirs = {
    "DrugEx_GT_epsilon_0.6": "DrugEx_GT_epsilon_0.6_10k",
    "GB_GA_mut_r_0.5": "GB_GA_mut_r_0.5_10k",
}

expected_files = [
    file_path
    for directory in dataset_dirs.values()
    for file_path in (data_root / directory).glob("*_dis_*_one_column.csv")
]

if len(expected_files) < 10:
    url = "https://zenodo.org/records/18678356/files/data.zip?download=1"
    with fsspec.open(url, "rb") as archive_handle, zipfile.ZipFile(archive_handle) as archive:
        selected = [
            name for name in archive.namelist()
            if name.startswith("data/output_sets/Glucocorticoid_receptor/")
            and any(directory in name for directory in dataset_dirs.values())
            and name.endswith("_one_column.csv")
        ]
        for name in selected:
            archive.extract(name, "zenodo_data")

In [ ]:
def load_generated_set(directory: str, generator: str) -> pd.DataFrame:
    files = sorted(glob(str(data_root / directory / "*_dis_*_one_column.csv")))
    if not files:
        raise FileNotFoundError(f"No input files found in {data_root / directory}")

    frames = []
    for file_name in files:
        run = Path(file_name).stem.split("_dis_")[1].split("_")[0]
        frame = pd.read_csv(file_name, header=None, names=["SMILES"])
        frames.append(frame.assign(run=run, generator=generator))
    return pd.concat(frames, ignore_index=True)

drugex = load_generated_set(dataset_dirs["DrugEx_GT_epsilon_0.6"], "DrugEx_GT_epsilon_0.6")
gbga = load_generated_set(dataset_dirs["GB_GA_mut_r_0.5"], "GB_GA_mut_r_0.5")

for name, frame in {"DrugEx": drugex, "GB-GA": gbga}.items():
    print(f"{name}: {len(frame):,} rows")
    display(frame.head())
    display(frame.groupby("run").size().rename("molecules"))


## Standardize structures and remove invalid molecules and duplicates


In [ ]:
def clean_smiles(smiles):
    if not isinstance(smiles, str):
        return None
    try:
        molecule = Chem.MolFromSmiles(smiles)
        if molecule is None:
            return None
        molecule = rdMolStandardize.FragmentParent(molecule)
        if molecule is None or molecule.GetNumAtoms() == 0:
            return None
        return Chem.MolToSmiles(molecule, canonical=True, isomericSmiles=True)
    except Exception:
        return None

def clean_dataset(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    cleaned = frame.copy()
    original_count = len(cleaned)
    cleaned["SMILES"] = cleaned["SMILES"].map(clean_smiles)
    invalid_count = cleaned["SMILES"].isna().sum()
    cleaned = cleaned.dropna(subset=["SMILES"])
    duplicate_count = cleaned.duplicated(subset=["SMILES"]).sum()
    cleaned = cleaned.drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
    print(
        f"{name}: {original_count:,} input, {invalid_count:,} invalid, "
        f"{duplicate_count:,} duplicates, {len(cleaned):,} retained"
    )
    return cleaned

drugex = clean_dataset(drugex, "DrugEx")
gbga = clean_dataset(gbga, "GB-GA")

# Keep origin labels and retain cross-origin overlaps: their overlap is informative.
shared_smiles = set(drugex["SMILES"]) & set(gbga["SMILES"])
print(f"Molecules shared by both origins: {len(shared_smiles):,}")

generated = pd.concat([drugex, gbga], ignore_index=True)
display(generated.groupby("generator").size().rename("molecules"))


## Physicochemical descriptor representation


In [ ]:
descriptor_columns = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "FractionCSP3",
]

def calc_descriptors(smiles: str) -> dict:
    molecule = Chem.MolFromSmiles(smiles)
    return {
        "MolWt": Descriptors.MolWt(molecule),
        "MolLogP": Crippen.MolLogP(molecule),
        "TPSA": rdMolDescriptors.CalcTPSA(molecule),
        "HBD": Lipinski.NumHDonors(molecule),
        "HBA": Lipinski.NumHAcceptors(molecule),
        "RotatableBonds": Lipinski.NumRotatableBonds(molecule),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(molecule),
    }

descriptor_values = generated["SMILES"].map(calc_descriptors).apply(pd.Series)
generated = pd.concat([generated, descriptor_values], axis=1)
display(generated.groupby("generator")[descriptor_columns].describe())


In [ ]:
# Save labels, structures, and descriptors without writing a synthetic index column.
descriptor_output = Path("chemical_space_descriptors.csv")
generated[["SMILES", "run", "generator", *descriptor_columns]].to_csv(
    descriptor_output, index=False
)
print(f"Saved {descriptor_output}")


## PCA of the descriptor representation

The descriptor columns are selected explicitly so that row indices, labels, and constant validity flags cannot enter the model. Missing values are imputed and the non-binary descriptors are standardized jointly across both origins.


In [ ]:
X_descriptors = generated[descriptor_columns].replace([np.inf, -np.inf], np.nan)
descriptor_preprocessor = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
X_descriptors_scaled = descriptor_preprocessor.fit_transform(X_descriptors)

pca = PCA(n_components=2)
pca_coordinates = pca.fit_transform(X_descriptors_scaled)

pca_plot = generated[["generator", "run"]].copy()
pca_plot[["PC1", "PC2"]] = pca_coordinates

fig, ax = plt.subplots(figsize=(8, 6))
for generator, group in pca_plot.groupby("generator"):
    ax.scatter(
        group["PC1"],
        group["PC2"],
        label=generator,
        alpha=0.35,
        s=12,
        linewidths=0,
    )
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("PCA of standardized physicochemical descriptors")
ax.legend(title="Dataset origin")
fig.tight_layout()
plt.show()


## Next representation

Morgan fingerprints (radius 2, 2048 bits) and t-SNE remain the next steps. As with PCA, the two origins should be embedded together and colored using the retained `generator` column.
